# Supplementary Embedding Plots

Two analysis sections:

1. **Supplementary Plots** — joint vs single-modality encoding similarity; normal vs full-run segment comparison
2. **Embedding Similarity Analysis** — RDMs, audio-video cosine similarity, and diagonal vs off-diagonal statistics across PE-AV model variants

**Environment:** `analysis`  
**Embedding path convention:** `{OUTPUT_BASE}/{model_name}/{bin_sec}s/{model_name}_{v,a,av}.npy`

In [ ]:
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.signal import fftconvolve
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import gamma
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
OUTPUT_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
TIMING_CSV = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")

BIN_SEC = 2.0   # seconds per embedding bin
ALPHA   = 0.05  # significance threshold

# PE-AV model variants to compare
PEAV_MODELS = [
    "pe-av-small-16-frame",
    "pe-av-small",
    "pe-av-base",
    "pe-av-base-16-frame",
    "pe-av-large",
    "pe-av-large-16-frame",
]
# ─────────────────────────────────────────────────────────────────────────────

timing_df = pd.read_csv(TIMING_CSV)
print(f"Timing: {len(timing_df)} video segments")

## Helper Functions

In [ ]:
def load_embeddings(model_name: str, modality: str, bin_sec: float) -> np.ndarray:
    """Load a single-modality embedding array.

    Args:
        model_name: str — directory name under OUTPUT_BASE
        modality:   str — "v", "a", or "av"
        bin_sec:    float — temporal bin size in seconds

    Returns:
        (n_bins, n_features) float32
    """
    path = OUTPUT_BASE / model_name / f"{int(bin_sec)}s" / f"{model_name}_{modality}.npy"
    if not path.exists():
        raise FileNotFoundError(f"Embedding not found: {path}")
    return np.load(path).astype(np.float32)


def compute_rdm(embeddings: np.ndarray, metric: str = "correlation") -> np.ndarray:
    """Compute a representational dissimilarity matrix.

    Args:
        embeddings: (n_bins, n_features)
        metric:     "correlation" (1 - Pearson r) or "cosine"

    Returns:
        (n_bins, n_bins) float64 symmetric RDM
    """
    return squareform(pdist(embeddings.astype(np.float64), metric=metric))


def spm_hrf(tr: float, oversampling: int = 16) -> np.ndarray:
    """SPM canonical HRF sampled at the given TR."""
    dt = tr / oversampling
    t  = np.arange(0, 32 + dt, dt)
    hrf = gamma.pdf(t, 6) - gamma.pdf(t, 16) / 6.0
    hrf = hrf[::oversampling]
    hrf /= hrf.sum()
    return hrf


def compute_av_similarity(audio: np.ndarray, video: np.ndarray) -> np.ndarray:
    """Cosine similarity matrix between audio and video embeddings."""
    return cosine_similarity(audio.astype(np.float64), video.astype(np.float64))


def compute_diag_stats(sim: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Diagonal value and per-row off-diagonal mean ± SE."""
    diag     = np.diag(sim)
    mask     = ~np.eye(sim.shape[0], dtype=bool)
    off_mean = np.array([sim[i, mask[i]].mean() for i in range(sim.shape[0])])
    off_se   = np.array([stats.sem(sim[i, mask[i]])  for i in range(sim.shape[0])])
    return diag, off_mean, off_se


def find_significant_segments(sim: np.ndarray, alpha: float = 0.05) -> List[int]:
    """Indices where diagonal cosine similarity > off-diagonal (one-sided t-test)."""
    sig = []
    for i in range(sim.shape[0]):
        off_vals = np.concatenate([sim[i, :i], sim[i, i + 1:]])
        _, p = stats.ttest_1samp(off_vals, sim[i, i], alternative="less")
        if p < alpha and sim[i, i] > off_vals.mean():
            sig.append(i)
    return sig


# ── Plotting helpers ───────────────────────────────────────────────────────────

def plot_heatmap(matrix: np.ndarray, title: str,
                xlabel: str = "Segment", ylabel: str = "Segment",
                cmap: str = "coolwarm", label: str = "Value") -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, origin="upper", cmap=cmap,
                   vmin=np.nanmin(matrix), vmax=np.nanmax(matrix))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=label)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def plot_rdm(rdm: np.ndarray, title: str) -> None:
    plot_heatmap(rdm, title, cmap="viridis", label="Dissimilarity (1 - r)")


def plot_diag_stats(diag, off_mean, off_se, sig_indices, title, alpha) -> None:
    x = np.arange(len(diag))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, diag, label="Diagonal (matched A/V)", linewidth=1.5)
    ax.plot(x, off_mean, label="Off-diagonal mean", linewidth=1.5, color="C1")
    ax.fill_between(x, off_mean - off_se, off_mean + off_se,
                    color="C1", alpha=0.3, label="Off-diagonal ±1 SE")
    if sig_indices:
        ax.scatter(sig_indices, diag[sig_indices], marker="*",
                   color="red", s=150, zorder=3, label=f"Significant (p < {alpha})")
    ax.set_xlabel("Segment index", fontsize=11)
    ax.set_ylabel("Cosine similarity", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


print("Helpers defined.")

## Supplementary Plots

Compare **joint** vs **single-modality** PE-AV encodings, and **normal** (chunked segments) vs **full-run** embeddings.

In [ ]:
# ── Joint vs single-modality comparison ───────────────────────────────────────
MODEL = "pe-av-small-16-frame"

audio_joint = load_embeddings(MODEL, "a", BIN_SEC)
video_joint = load_embeddings(MODEL, "v", BIN_SEC)
print(f"Joint — audio {audio_joint.shape}, video {video_joint.shape}")

sim_joint = compute_av_similarity(audio_joint, video_joint)
print(f"Similarity: mean={sim_joint.mean():.3f}  std={sim_joint.std():.3f}")

plot_heatmap(sim_joint, f"{MODEL}: A/V Cosine Similarity (joint encoding)",
             xlabel="Video segment", ylabel="Audio segment", cmap="coolwarm",
             label="Cosine similarity")

In [ ]:
# ── RSA between joint and single encodings ─────────────────────────────────────
# Compares upper-triangle of the two similarity matrices
sim_flat = sim_joint[np.triu_indices_from(sim_joint, k=1)]

# Placeholder: load single-modality if available
single_path = OUTPUT_BASE / MODEL / "pe-av-small-16-frame-single-modality"
if (single_path / f"{MODEL}_a.npy").exists():
    audio_single = np.load(single_path / f"{MODEL}_a.npy").astype(np.float64)
    video_single = np.load(single_path / f"{MODEL}_v.npy").astype(np.float64)
    sim_single = compute_av_similarity(audio_single, video_single)
    sim_single_flat = sim_single[np.triu_indices_from(sim_single, k=1)]

    rho, p = stats.spearmanr(sim_flat, sim_single_flat)
    print(f"RSA (joint vs single): Spearman ρ = {rho:.4f}  p = {p:.2e}")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(sim_flat, sim_single_flat, alpha=0.3, s=8)
    ax.plot([-1, 1], [-1, 1], "r--", linewidth=1.5, label="Identity")
    ax.set_xlabel("Joint encoding similarity", fontsize=11)
    ax.set_ylabel("Single-modality similarity", fontsize=11)
    ax.set_title(f"Joint vs Single — Spearman ρ = {rho:.4f}", fontsize=12)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print(f"Single-modality embeddings not found at {single_path} — skipping comparison.")

## Embedding Similarity Analysis

For each PE-AV variant, compute:
- Audio-video cosine similarity matrix
- Audio, video, and joint RDMs
- Diagonal vs off-diagonal statistics (matched vs unmatched segment pairs)

In [ ]:
for model_name in PEAV_MODELS:
    emb_dir = OUTPUT_BASE / model_name / f"{int(BIN_SEC)}s"
    if not emb_dir.exists():
        print(f"SKIP {model_name} — embeddings not found")
        continue

    print(f"\n{'='*60}")
    print(f"{model_name}")
    print(f"{'='*60}")

    try:
        audio = load_embeddings(model_name, "a", BIN_SEC)
        video = load_embeddings(model_name, "v", BIN_SEC)
        av    = load_embeddings(model_name, "av", BIN_SEC)
    except FileNotFoundError as e:
        print(f"  SKIP — {e}")
        continue

    print(f"  audio {audio.shape}  video {video.shape}  av {av.shape}")

    # Audio-video cosine similarity
    sim = compute_av_similarity(audio, video)
    plot_heatmap(sim, f"{model_name}: A/V Cosine Similarity",
                 xlabel="Video segment", ylabel="Audio segment",
                 cmap="coolwarm", label="Cosine similarity")

    # RDMs
    for mod_name, emb in [("Audio", audio), ("Video", video), ("AV", av)]:
        rdm = compute_rdm(emb)
        plot_rdm(rdm, f"{model_name}: {mod_name} RDM")

    # Diagonal vs off-diagonal analysis
    diag, off_mean, off_se = compute_diag_stats(sim)
    sig = find_significant_segments(sim, alpha=ALPHA)
    print(f"  Diagonal: mean={diag.mean():.3f}  Off-diag: mean={off_mean.mean():.3f}")
    print(f"  Significant (diag > off-diag, p < {ALPHA}): {len(sig)} / {len(diag)}")
    plot_diag_stats(diag, off_mean, off_se, sig,
                    f"{model_name}: Diagonal vs Off-diagonal Similarity", ALPHA)

print("\nDone.")